In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 모델 초기화
model = init_chat_model(
    "google_genai:gemini-3.1-flash-lite",
    temperature=0
)

In [2]:
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import HumanMessage, AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

# State 정의
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# Node 정의
def llm_node(state: MessagesState):
    response = model.invoke(
        state["messages"]
    )
    return {"messages": [response]}

# Graph 생성
graph_builder = StateGraph(MessagesState)

# Graph에 Node 추가
graph_builder.add_node("llm", llm_node)

# Edge 추가하여 Node 연결
graph_builder.add_edge(START, "llm")
graph_builder.add_edge("llm", END)

# Graph를 실행 가능한 형태로 컴파일
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [3]:
config = {"configurable": {"thread_id": "conversation_1"}}

In [4]:
human_message = HumanMessage(content="내 이름은 김일남이야.")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름은 김일남이야.
================================== Ai Message ==================================

[{'type': 'text', 'text': '반갑습니다, 김일남 님! 만나서 정말 기뻐요. 오늘 하루는 어떻게 보내고 계신가요? 제가 도와드릴 일이 있다면 무엇이든 편하게 말씀해 주세요!', 'extras': {'signature': 'EjQKMgEMOdbHjITR+pdAdoqQx1GKx6FKYLzA0NbE54V0J5F1w3K7SOb2oDe1EOw3Iq9l98nL'}}]


In [5]:
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름은 김일남이야.
================================== Ai Message ==================================

[{'type': 'text', 'text': '반갑습니다, 김일남 님! 만나서 정말 기뻐요. 오늘 하루는 어떻게 보내고 계신가요? 제가 도와드릴 일이 있다면 무엇이든 편하게 말씀해 주세요!', 'extras': {'signature': 'EjQKMgEMOdbHjITR+pdAdoqQx1GKx6FKYLzA0NbE54V0J5F1w3K7SOb2oDe1EOw3Iq9l98nL'}}]
================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

[{'type': 'text', 'text': '방금 말씀해 주셨죠! 당신의 이름은 **김일남** 님입니다. 기억하고 있어요! :)', 'extras': {'signature': 'EjQKMgEMOdbHtxGR7a+8jn/yOfcNCiAHdeCbSLAETZulWIjTsX0kOoFNLHmE/4N1aJYtV81Z'}}]


In [6]:
config = {"configurable": {"thread_id": "conversation_2"}}

In [7]:
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

[{'type': 'text', 'text': '죄송하지만, 저는 사용자의 개인정보를 저장하거나 기억하지 않기 때문에 당신의 이름을 알지 못합니다. \n\n혹시 이전에 저에게 이름을 알려주셨더라도, 새로운 대화 세션이 시작되면 이전 대화의 내용을 기억할 수 없습니다. 원하신다면 지금 이름을 알려주세요! 기억해 두겠습니다.', 'extras': {'signature': 'EjQKMgEMOdbHQVmxSw+kmxLIdvRXizq1kKEvB4MTCd8f4ltegSYUdb8WuT5JDdvR9h/YKbRK'}}]
